@author: eveomett

# Lab 3: MAUP and data.  See details on Canvas page

## Make sure to say where/when you got your data!

In [1]:
import pandas as pd
import geopandas as gpd
import maup
from maup import smart_repair
import time
import os

maup.progress.enabled = True

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
# paths
population_path = ".\ma_pl2020_b\ma_pl2020_p1_b.shp"
election_path = ".\ma_2024_gen_prec\ma_2024_gen_all_prec\ma_2024_gen_all_prec.shp"
county_path = ".\ma_pl2020_cnty\ma_pl2020_cnty.shp"

# load data
pop_df = gpd.read_file(population_path)
county_df = gpd.read_file(county_path)
elec_df = gpd.read_file(election_path)

print("population rows:", len(pop_df))
print("county rows:", len(county_df))
print("election rows:", len(elec_df))

population rows: 107278
county rows: 14
election rows: 2383


In [4]:
# move everything to one UTM CRS
utm_crs = elec_df.estimate_utm_crs()

pop_df = pop_df.to_crs(utm_crs)
county_df = county_df.to_crs(utm_crs)
elec_df = elec_df.to_crs(utm_crs)

print(pop_df.crs)
print(county_df.crs)
print(elec_df.crs)

# doctor checks before repair
print("population doctor:", maup.doctor(pop_df))
print("county doctor:", maup.doctor(county_df))
print("election doctor:", maup.doctor(elec_df))

EPSG:32619
EPSG:32619
EPSG:32619


100%|█████████████████████████████████████████████████████████████████████████| 107278/107278 [08:15<00:00, 216.68it/s]


population doctor: True


100%|██████████████████████████████████████████████████████████████████████████████████| 14/14 [00:00<00:00, 26.36it/s]


county doctor: True


100%|█████████████████████████████████████████████████████████████████████████████| 2383/2383 [00:20<00:00, 118.72it/s]


There are 519 overlaps.
There are 77 holes.
There are some invalid geometries.
election doctor: False


In [5]:
# repair precincts inside counties
final_df = smart_repair(elec_df, nest_within_regions = county_df, snap_precision = 8)

# change small rook boundaries to queen adjacencies
final_df = smart_repair(final_df, min_rook_length = 30)

print("final doctor:", maup.doctor(final_df))

100%|██████████████████████████████████████████████████████████████████████████████████| 14/14 [00:00<00:00, 25.30it/s]


Snapping all geometries to a grid with precision 10^( -3 ) to avoid GEOS errors.


100%|██████████████████████████████████████████████████████████████████████████████████| 14/14 [00:02<00:00,  5.89it/s]


Identifying overlaps...


100%|█████████████████████████████████████████████████████████████████████████████| 8753/8753 [00:28<00:00, 307.68it/s]


...

Resolving overlaps...
Filling gaps...


Gaps to simplify: 0it [00:00, ?it/s]
Gaps to fill: 0it [00:00, ?it/s]


Converting small rook adjacencies to queen...


100%|█████████████████████████████████████████████████████████████████████████████| 2383/2383 [00:17<00:00, 139.00it/s]


final doctor: True


In [ ]:
# assign 2020 blocks to precincts and add total population
precinct_assignments = maup.assign(pop_df.geometry, final_df.geometry)

final_df["TOTPOP"] = pop_df["P0010001"].groupby(precinct_assignments).sum()
final_df["TOTPOP"] = final_df["TOTPOP"].fillna(0)

# keep district data from the precinct file
final_df["CD"] = final_df["CONG_DIST"]

# save shapefile
if not os.path.exists("MA"):
    os.mkdir("MA")

final_df.to_file(r"MA\MA.shp")

print("STACK SUMMARY")
print("precinct file used:", election_path)
print("population file used:", population_path)
print("county file used:", county_path)
print("doctor after repair:", maup.doctor(final_df))
print("total population in blocks:", pop_df["P0010001"].sum())
print("total population in final precincts:", final_df["TOTPOP"].sum())
print("number of districts:", final_df["CD"].nunique())
print(final_df["CD"].dropna().unique())

 43%|████████████████████████████████▉                                            | 1020/2383 [00:07<00:10, 126.37it/s]

In [ ]:
# gerrychain imports
import networkx as nx
import gerrychain
from gerrychain import Graph, Partition, constraints, MarkovChain
from gerrychain.updaters import cut_edges, Tally
from gerrychain.proposals import recom
from gerrychain.accept import always_accept
from functools import partial
from gerrychain.constraints import contiguous

In [ ]:
# build graph and initial partition
ma_graph = Graph.from_file(r"MA\MA.shp", adjacency = "queen", cols_to_add = ["TOTPOP", "CD"])

print("Is graph connected?", nx.is_connected(ma_graph))
print("Number of nodes:", len(ma_graph.nodes()))
print("Number of edges:", len(ma_graph.edges()))

tot_pop = sum([ma_graph.nodes()[v]["TOTPOP"] for v in ma_graph.nodes()])
num_dist = len(set([ma_graph.nodes()[v]["CD"] for v in ma_graph.nodes()]))

ideal_pop = tot_pop / num_dist
pop_tolerance = 0.05

initial_plan = {v: ma_graph.nodes()[v]["CD"] for v in ma_graph.nodes()}

initial_partition = Partition(
    ma_graph,
    assignment = initial_plan,
    updaters = {
        "cut_edges": cut_edges,
        "district population": Tally("TOTPOP", alias = "district population")
    }
)

print("initial partition contiguous?", contiguous(initial_partition))

for d in sorted(set(initial_plan.values())):
    verts_in_dist = [v for v in ma_graph.nodes() if initial_plan[v] == d]
    district = nx.subgraph(ma_graph, verts_in_dist)
    print("district", d, "connected?", nx.is_connected(district), "components:", nx.number_connected_components(district))

In [ ]:
# optional map check to see disconnected pieces
df = gpd.read_file(r"MA\MA.shp")
df.index = list(ma_graph.nodes())

comp_id = {}
for i, comp in enumerate(nx.connected_components(ma_graph)):
    for v in comp:
        comp_id[v] = i

df["component"] = pd.Series(comp_id)

df.plot(column = "component", cmap = "tab20", figsize = (12,8), legend = True)

for d in ["07", "08", "09"]:
    df[df["CD"] == d].plot(column = "component", cmap = "tab20", figsize = (6,6), legend = True)
    print("district", d)

In [ ]:
# patch the graph by connecting the nearest disconnected pieces inside each district
temp_graph = nx.Graph(ma_graph)
centroids = df.geometry.centroid

for d in sorted(set(initial_plan.values())):
    verts_in_dist = [v for v in temp_graph.nodes() if initial_plan[v] == d]
    district = nx.subgraph(temp_graph, verts_in_dist)

    while not nx.is_connected(district):
        comps = sorted(nx.connected_components(district), key = len, reverse = True)
        main_comp = list(comps[0])
        other_comp = list(comps[1])

        best_u = None
        best_v = None
        best_dist = float("inf")

        for u in main_comp:
            for v in other_comp:
                dist = centroids.loc[u].distance(centroids.loc[v])
                if dist < best_dist:
                    best_dist = dist
                    best_u = u
                    best_v = v

        temp_graph.add_edge(best_u, best_v)
        print("added edge in district", d, "between", best_u, "and", best_v, "distance", round(best_dist, 2))

        district = nx.subgraph(temp_graph, verts_in_dist)

# put the needed node data back on the graph explicitly
for v in temp_graph.nodes():
    temp_graph.nodes[v]["TOTPOP"] = df.loc[v, "TOTPOP"]
    temp_graph.nodes[v]["CD"] = df.loc[v, "CD"]

ma_graph = gerrychain.Graph(temp_graph)

initial_partition = Partition(
    ma_graph,
    assignment = initial_plan,
    updaters = {
        "cut_edges": cut_edges,
        "district population": Tally("TOTPOP", alias = "district population")
    }
)

print("graph connected now?", nx.is_connected(ma_graph))
print("initial partition contiguous now?", contiguous(initial_partition))

print(ma_graph.nodes[list(ma_graph.nodes())[0]])

In [ ]:
# short random walk
rw_proposal = partial(
    recom,
    pop_col = "TOTPOP",
    pop_target = ideal_pop,
    epsilon = pop_tolerance,
    node_repeats = 1
)

population_constraint = constraints.within_percent_of_ideal_population(
    initial_partition,
    pop_tolerance,
    pop_key = "district population"
)

our_random_walk = MarkovChain(
    proposal = rw_proposal,
    constraints = [constraints.contiguous, population_constraint],
    accept = always_accept,
    initial_state = initial_partition,
    total_steps = 10
)

cutedge_ensemble = []

for part in our_random_walk:
    cutedge_ensemble.append(len(part["cut_edges"]))

print(cutedge_ensemble)

In [ ]:
# final summary
print("FINAL SUMMARY")
print("final shapefile doctor:", maup.doctor(final_df))
print("block population total:", pop_df["P0010001"].sum())
print("precinct population total:", final_df["TOTPOP"].sum())
print("district count:", final_df["CD"].nunique())
print("graph connected:", nx.is_connected(ma_graph))
print("initial partition contiguous:", contiguous(initial_partition))
print("cut edges from short run:", cutedge_ensemble)